In [14]:
from pathlib import Path
import re

root_folder = Path(r"D:\EIS-Data_9_10_26")

# Measurement order
frequencies = [1, 10, 100, 1000, 10000, 100000]

# Keep False first to preview names without renaming
DO_RENAME = False


for device_folder in root_folder.iterdir():

    if not device_folder.is_dir():
        continue

    # Example: 4_1
    device_name = device_folder.name

    for material_folder in device_folder.iterdir():

        if not material_folder.is_dir():
            continue

        # Example: Al_50
        material_name = material_folder.name

        # Find all channel-1 files
        ch1_files = list(material_folder.glob("*_Ch1.csv"))

        # Sort according to the numeric Detail number
        def detail_number(path):
            match = re.search(r"Detail(\d+)", path.name)
            return int(match.group(1)) if match else 0

        ch1_files.sort(key=detail_number)

        if len(ch1_files) != len(frequencies):
            print(
                f"WARNING: {material_folder} has "
                f"{len(ch1_files)} Ch1 files, expected "
                f"{len(frequencies)}."
            )
            continue

        for frequency, ch1_file in zip(frequencies, ch1_files):

            # Find corresponding Ch2 file
            ch2_file = ch1_file.with_name(
                ch1_file.name.replace("_Ch1.csv", "_Ch2.csv")
            )

            if not ch2_file.exists():
                print(f"WARNING: Missing Ch2 for {ch1_file.name}")
                continue

            # Desired naming scheme:
            # 4_1_Al_50_1_1.csv
            # 4_1_Al_50_1_2.csv

            new_ch1 = material_folder / (
                f"{device_name}_{material_name}_{frequency}_1.csv"
            )

            new_ch2 = material_folder / (
                f"{device_name}_{material_name}_{frequency}_2.csv"
            )

            print(f"{ch1_file.name} -> {new_ch1.name}")
            print(f"{ch2_file.name} -> {new_ch2.name}")

            if DO_RENAME:
                ch1_file.rename(new_ch1)
                ch2_file.rename(new_ch2)